---
title: "深度优先搜索 (DFS)——树型搜索"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [9]:
from collections import deque
from typing import Optional
class TreeNode:
    def __init__(self, val: int = 0, left: 'TreeNode' = None, right: 'TreeNode' = None):
        self.val = val
        self.left = left
        self.right = right

## 📝 题目 111：二叉树的最小深度

::: {.callout-caution}
### 题目要求
**描述**：
给定一个二叉树，找出其最小深度。
**最小深度**是从根节点到最近 **叶子节点** 的最短路径上的节点数量。
- **注意**：叶子节点是指没有左孩子，且没有右孩子的节点。

**输入输出示例**：

* **示例 1**：
    * **输入**：`root = [3, 9, 20, None, None, 15, 7]`
    * **输出**：`2`
    * **解释**：根节点 3 到左侧叶子节点 9 的路径长度为 2，这是最短的。

* **示例 2**：
    * **输入**：`root = [2, None, 3, None, 4, None, 5, None, 6]`
    * **输出**：`5`
    * **解释**：这是一棵极其不平衡的树。虽然根节点 2 的左边是空的，但因为 2 还有右孩子，所以它不是叶子。你必须一直走到最底层的 6 才是叶子节点。
:::

In [3]:
class Solution111:
    def minDepth(self, root: Optional[TreeNode]) -> int:
        if not root:
            return 0
        if root.left == None:
            return self.minDepth(root.right) + 1
        elif root.right == None:
            return self.minDepth(root.left) + 1
        return min(self.minDepth(root.left), self.minDepth(root.right)) + 1

In [4]:
#| code-fold: true
def build_tree(data):
    """将 LeetCode 格式的列表转换为二叉树对象"""
    if not data: return None
    root = TreeNode(data[0])
    queue = deque([root])
    i = 1
    while queue and i < len(data):
        curr = queue.popleft()
        if i < len(data) and data[i] is not None:
            curr.left = TreeNode(data[i])
            queue.append(curr.left)
        i += 1
        if i < len(data) and data[i] is not None:
            curr.right = TreeNode(data[i])
            queue.append(curr.right)
        i += 1
    return root

def test_111(func):
    test_cases = [
        {"input": [3, 9, 20, None, None, 15, 7], "expected": 2, "desc": "平衡树（最短在左）"},
        {"input": [2, None, 3, None, 4, None, 5, None, 6], "expected": 5, "desc": "右斜树（陷阱测试）"},
        {"input": [1, 2, None, 3, None, 4, None, 5], "expected": 5, "desc": "左斜树（陷阱测试）"},
        {"input": [1, 2, 3], "expected": 2, "desc": "简单满二叉树"},
        {"input": [], "expected": 0, "desc": "空树测试"},
        {"input": [1], "expected": 1, "desc": "单节点测试"},
        {"input": [1, 2, None], "expected": 2, "desc": "根节点只有左孩子"},
        {"input": [1, None, 2], "expected": 2, "desc": "根节点只有右孩子"}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 60)
    for i, tc in enumerate(test_cases):
        root = build_tree(tc["input"]) # 使用之前定义的层序建树函数
        actual = func(root)
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 60)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_111(Solution111().minDepth)

ID   | 结果     | 预期   | 实际   | 描述
------------------------------------------------------------
1    | ✅ PASS | 2    | 2    | 平衡树（最短在左）
2    | ✅ PASS | 5    | 5    | 右斜树（陷阱测试）
3    | ✅ PASS | 5    | 5    | 左斜树（陷阱测试）
4    | ✅ PASS | 2    | 2    | 简单满二叉树
5    | ✅ PASS | 0    | 0    | 空树测试
6    | ✅ PASS | 1    | 1    | 单节点测试
7    | ✅ PASS | 2    | 2    | 根节点只有左孩子
8    | ✅ PASS | 2    | 2    | 根节点只有右孩子
------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

这道题最容易犯的错误是直接套用最大深度的 `min(left, right) + 1`。我们需要通过分类讨论来处理那些“只有一条腿”的非叶子节点。

**1. 递归基准 (Base Case)**：
   - `if not root: return 0` (空节点深度为 0)。

**2. 核心递归逻辑（关键差异点）**：
   - **情况 A：左子树为空，右子树不为空**
     - 此时根节点不是叶子。最小深度必须在右边产生。
     - **动作**：返回 `self.minDepth(root.right) + 1`。
   - **情况 B：右子树为空，左子树不为空**
     - 此时根节点不是叶子。最小深度必须在左边产生。
     - **动作**：返回 `self.minDepth(root.left) + 1`。
   - **情况 C：左右子树都不为空**
     - 此时两边都有路，我们才真正需要对比哪条路更短。
     - **动作**：返回 `min(self.minDepth(root.left), self.minDepth(root.right)) + 1`。
   - **情况 D：左右子树都为空（叶子节点）**
     - 会触发前两个判断之一或直接计算，返回 `0 + 1 = 1`。

**3. 避坑指南**：
   - 记住：如果一个节点只有一个孩子，它**不是叶子节点**。你不能取它那个空孩子（深度0）作为最小值，否则你会得到错误的答案 1。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。其中 $N$ 是二叉树中的节点个数，每个节点仅被访问一次。
* **空间复杂度**：$O(H)$。其中 $H$ 是树的高度。在最坏情况下（树退化成链表），空间复杂度为 $O(N)$；在平衡树情况下为 $O(\log N)$。
:::

## 📝 题目 112：路径总和 (Path Sum)

::: {.callout-caution}
### 题目要求
**描述**：
给你二叉树的根节点 `root` 和一个表示目标和的整数 `targetSum` 。
判断该树中是否存在 **根节点到叶子节点** 的路径，这条路径上所有节点值相加等于目标和 `targetSum` 。

**注意**：
- **叶子节点** 是指没有左孩子且没有右孩子的节点。
- 路径必须从根节点开始，到叶子节点结束。

**输入输出示例**：

* **示例 1**：
    * **输入**：`root = [5,4,8,11,None,13,4,7,2,None,None,None,1], targetSum = 22`
    * **输出**：`True`
    * **解释**：存在路径 5 -> 4 -> 11 -> 2，总和为 22。

* **示例 2**：
    * **输入**：`root = [1,2,3], targetSum = 5`
    * **输出**：`False`
    * **解释**：树中只有两条路径：1->2 (和为3) 和 1->3 (和为4)。没有和为 5 的路径。
:::

In [5]:
class Solution112:
    def hasPathSum(self, root: Optional[TreeNode], targetSum: int) -> bool:
        if not root:  # 走到空地了
            return False
        if not root.left and not root.right:  # 到达叶子节点
            return root.val == targetSum
        remain = targetSum - root.val  # 还没到终点，继续往下走（递归）
        return self.hasPathSum(root.left, remain) or self.hasPathSum(root.right, remain)  # 拿着剩下的任务，分头行动

In [6]:
#| code-fold: true
def test_112(func):
    test_cases = [
        {"input": [10, 5, 12, 3], "target": 18, "expected": True, "desc": "3层小树存在路径"},
        {"input": [10, 5, 12, 3], "target": 22, "expected": True, "desc": "右侧路径 10->12"},
        {"input": [1, 2, 3], "target": 5, "expected": False, "desc": "路径和不匹配"},
        {"input": [], "target": 0, "expected": False, "desc": "空树"},
        {"input": [1, 2], "target": 1, "expected": False, "desc": "根节点不是叶子"}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<5} | {'实际':<5} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        root = build_tree(tc["input"])
        actual = func(root, tc["target"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {str(tc['expected']):<5} | {str(actual):<5} | {tc['desc']}")
    print("-" * 65)
    print(f"总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_112(Solution112().hasPathSum)

ID   | 结果     | 预期    | 实际    | 描述
-----------------------------------------------------------------
1    | ✅ PASS | True  | True  | 3层小树存在路径
2    | ✅ PASS | True  | True  | 右侧路径 10->12
3    | ✅ PASS | False | False | 路径和不匹配
4    | ✅ PASS | False | False | 空树
5    | ✅ PASS | False | False | 根节点不是叶子
-----------------------------------------------------------------
总结: 通过 5/5


::: {.callout-important}
### 💡 思路讲解

与其在路径上不停地“加”值，不如反过来用“减”法，这样处理起来更清爽。

**1. 任务下达（递归逻辑）**：
   - 根节点对孩子说：“我的目标是 `targetSum`，但我占用了 `root.val`。你们谁能在我剩下的值 `targetSum - root.val` 里找到凑成 0 的路径？”

**2. 递归基准 (Base Case)**：
   - **空地**：`if not root: return False`（没路了，任务失败）。
   - **到达叶子节点（终点判定）**：
     - 如果当前节点是叶子（左右都为空），我们检查：它自己本身的值是否等于剩下的目标值？
     - `return root.val == targetSum`

**3. 结果汇总**：
   - 只要左子树 **或者** 右子树中有一条路径能达成目标，就返回 `True`。
   - `return self.hasPathSum(root.left, targetSum - root.val) or self.hasPathSum(root.right, targetSum - root.val)`
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。每个节点访问一次。
* **空间复杂度**：$O(H)$。取决于树的高度，用于递归调用栈。
:::

## 📝 题目 226：翻转二叉树

::: {.callout-caution}
### 题目要求
**描述**：
给你一棵二叉树的根节点 `root`，翻转这棵二叉树，并返回其根节点。

**直观理解**：
想象这棵树原本在照镜子，翻转后就是它在镜子里的样子。所有的左孩子变右孩子，右孩子变左孩子。

**输入输出示例**：

* **示例 1**：
    * **输入**：`root = [4, 2, 7, 1, 3, 6, 9]`
    * **输出**：`[4, 7, 2, 9, 6, 3, 1]`
    * **说明**：根节点 4 的左右孩子 (2, 7) 交换了位置，且它们底下的孙子节点也全部互相交换了位置。

* **示例 2**：
    * **输入**：`root = [2, 1, 3]`
    * **输出**：`[2, 3, 1]`

* **示例 3**：
    * **输入**：`root = []`
    * **输出**：`[]`
:::

In [7]:
class Solution226:
    def invertTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if not root:  # 如果是空节点，直接返回
            return None
        root.left, root.right = root.right, root.left
        self.invertTree(root.left)
        self.invertTree(root.right)
        return root

In [8]:
#| code-fold: true
def test_226(func):
    test_cases = [
        {"input": [4, 2, 7, 1, 3, 6, 9], "expected": [4, 7, 2, 9, 6, 3, 1], "desc": "标准满二叉树"},
        {"input": [2, 1, 3], "expected": [2, 3, 1], "desc": "三节点树"},
        {"input": [1, 2], "expected": [1, None, 2], "desc": "只有左孩子"},
        {"input": [], "expected": [], "desc": "空树"}
    ]

    # 辅助函数：将树转回数组格式方便对比
    def tree_to_list(root):
        if not root: return []
        res, q = [], [root]
        while q:
            node = q.pop(0)
            if node:
                res.append(node.val)
                q.append(node.left)
                q.append(node.right)
            else:
                res.append(None)
        while res and res[-1] is None: res.pop()
        return res

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 40)
    for i, tc in enumerate(test_cases):
        root = build_tree(tc["input"])
        actual_root = func(root)
        actual_list = tree_to_list(actual_root)
        is_pass = actual_list == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")
    print("-" * 40)
    print(f"总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_226(Solution226().invertTree)

ID   | 结果     | 描述
----------------------------------------
1    | ✅ PASS | 标准满二叉树
2    | ✅ PASS | 三节点树
3    | ✅ PASS | 只有左孩子
4    | ✅ PASS | 空树
----------------------------------------
总结: 通过 4/4


::: {.callout-important}
### 💡 思路讲解

这道题的递归逻辑可以拆解为两步：**“动自己的手”**和**“派孩子去干活”**。

**1. 递归基准情形 (Base Case)**：
   - 如果 `root` 是空的，直接返回 `None`。空地不需要翻转。

**2. 核心操作 (局部交换)**：
   - 想象你是一个节点（比如根节点 4）：
   - **交换**：把你左手拉着的 2 和右手拉着的 7 换一下位置。
   - 现在你手里变成了：左边是 7，右边是 2。

**3. 递归向下 (分发任务)**：
   - **左边去翻转**：指派左儿子 (7) 去翻转它自己那一支。
   - **右边去翻转**：指派右儿子 (2) 去翻转它自己那一支。
   - **注意**：不管你先交换还是先递归，最终只要保证每一个节点都把自己的左右孩子交换过一次，整棵树就翻转成功了。

**4. 结果汇报**：
   - 翻转完成后，把已经处理好的自己 (`root`) 返回给上一层。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。树中每个节点都要“交换”一次，总共 $N$ 个节点。
* **空间复杂度**：$O(H)$。取决于树的高度，用于递归调用栈。最坏情况 $O(N)$，平衡树 $O(\log N)$。
:::